# COGS 188 - Project Proposal

# Names

- Ranya Tashkandy
- Daria Stolyarova
- Rosa Gonzalez Encinas

# Abstract 
Many people believe that anyone who has been arrested should always be labeled and never trusted again. But a single label ignores that risk differs from person to person and changes with time after release. Estimating individual risk may provide more information than relying on a single label, although prediction alone does not establish fairness. This project asks: can we predict how likely someone is to be rearrested within three years of starting parole supervision? We use National Institute of Justice (NIJ) data on 25,835 people released from Georgia prisons in 2013–2015, with information about each person and whether they had a recorded rearrest within three years. We will use gradient descent to fit the logistic regression models and compare hill climbing with random-restarts and simulated annealing for choosing which predictors go into them. We will then test on held-out data using the Brier score (how close the predicted chances are to what happened) and ROC-AUC (how well the model ranks people with recorded rearrest above those without one), and compare against a logistic regression model that uses all the predictors we're considering.

# Background

A common belief is that once someone has been arrested, they are still the same person they were, or worse, when the crime was committed. This belief impacts a person's livelihood and shapes real decisions, since a majority of U.S. employers screen job applicants for criminal records, and many are unwilling to hire someone who has one. A field experiment showed how much having a criminal record can affect an applicant's chances of getting a callback or job offer. In it, matched pairs of job applicants applied for real entry-level jobs, and a criminal record was associated with a 50% drop in employment opportunities for white applicants and a 64% drop for Black applicants. <sup><a id="cite2" href="#note2">[2]</a></sup>

A label treats everyone with a record as equally risky, but research shows that risk is not fixed. Criminologists widely accept that recidivism declines steadily the longer someone stays arrest-free. <sup><a href="#note1">[1]</a></sup>National data shows the same pattern among people leaving prison: among prisoners released in 2012, the yearly arrest rate fell from 37% in Year 1 to 26% in Year 5.<sup><a id="cite3" href="#note3">[3]</a></sup>

This helps explain why a single label can miss differences in what happens after release. Our model will estimate each person’s chance of a recorded rearrest within three years of starting parole supervision. Our project will use search methods to see whether a smaller set of predictors can perform about as well as the full set. We will compare false-positive rates across groups to see whether the model predicts rearrest more often for one group among people who had no recorded rearrest.


# Problem Statement

For each person released from a Georgia prison onto parole, we know information about them, such as age, prior arrests, and prison offense. We also know whether they had a recorded rearrest within three years of starting parole supervision. The problem we are solving is to predict, for each person, the chance of rearrest within that three-year window using information avaibale at release. A machine learning solution is to fit a logistic regression model on the training set to learn how these predictors relate to the chance of rearrest.

This problem is quantifiable because it can be written as a number. For each person, the model gives one probability between 0 and 1, and the recorded outcome is 1 if the person was rearrested within three years and 0 otherwise. The goal is to make the predicted chances as close as possible to the recorded outcomes. Choosing which predictors to use is also quantifiable, because each subset gets a Brier score on the validation set. We will use hill climbing with random-restarts hill climbing and simulated annealing to search for subsets with lower scores.

The problem is measurable because we will score the models on held-out test set using Brier score and ROC-AUC. The Brier score measures how close the predicted chances are to what happened, where lower is better. ROC-AUC measures how well the model ranks people who were rearrested above people who were not, where higher is better. We compare with what we get when we fit logistic regression using all the eligible predictors.

The problem is replicable because the data is public and fixed. We’ll report the random seed for our stratified 60/20/20 training, validation, and test split, along with the search seeds and model settings, so anyone can repeat our steps and get comparable results.


# Data

For our data, we will use the full dataset from the [NIJ Recidivism Forecasting Challenge](https://nij.ojp.gov/funding/recidivism-forecasting-challenge), which contains 25,835 observations and 54 columns. Each row represents a person released from prison to parole supervision in Georgia between 2013 and 2015. The full dataset includes the original training and test records with outcome labels, so we will create our own splits without combining separate files. Our target variable will be 'Recidivism_Within_3years', which records whether someone was rearrested for a new felony or misdemeanor within three years of starting supervision. We will encode its 14,904 “Yes” outcomes as 1 and its 10,931 “No” outcomes as 0. Our predictions will concern recorded rearrest, which does not establish whether someone committed another crime.

We will consider predictors such as age at release, education, prison offense, time served, and previous arrests and convictions. These variables include grouped ranges, categorical labels, binary values, and counts with upper limits. Before choosing the final inputs, we will check the codebook to confirm when each variable was collected. Since we will make predictions using information available at release, we will exclude later information such as drug-test results, employment during parole, and supervision violations. We will also remove IDs, the original 'Training_Sample' indicator, and all outcome columns from the predictors. We will keep race separate from the predictors and use it only to compare errors across groups.

During preprocessing, we will fill missing numerical values with the training-set median and treat missing categorical values as a separate category. Categorical predictors, including age and time-served ranges, will be one-hot encoded, and numerical predictors will be standardized. All preprocessing will be fitted using only the training data, then applied to the validation and test sets. We plan to use a stratified 60/20/20 training, validation, and test split with a fixed random seed. The validation set will be used to compare feature subsets during hill climbing with random starts and simulated annealing, while the test set will remain separate until the models are finalized. Repeatedly using the test set to choose features could lead us to select a model that performs well on that particular data but does not perform as well on new observations. 

# Proposed Solution

Our baseline will be logistic regression using all eligible predictors. We will compare it with logistic regression models using features selected by hill climbing with random starts and simulated annealing. We chose logistic regression because it produces rearrest probabilities and should be practical to train repeatedly during feature selection. Our goal is to determine whether either search can identify a smaller set of predictors that performs as well as or better than the baseline.

Each candidate feautre set will use some of the eligible predictors. To get a neighboring set, we'll add or remove one predictor at a time. All one-hot encoded columns from a categorical predictor will stay together. Candidate models will be fitted on the training set and compared using validation Brier score, while the logistic regression settings will remain fixed across approaches.

Hill climbing will evaluate neighboring feature sets and move to the one with the greatest improvement in validation Brier score. When no move improves the validation Brier score, hill climbing will start from a new random set of predictors. The search will stop when it runs out of restarts, model fits, or search steps, whichever comes first. Simulated annealing will randomly add or remove one predictor at each step and sometimes accept a worse score to explore beyond a local optimum. The probability of accepting a worse change will follow $e^{-\frac{\Delta}{T}}$, where $Δ$ is the increase in Brier score and $T$ is the temperature. As the temperature decreases, the search will become less likely to accept worse changes. We will retain the best feature set found during each run.
    
Before the main comparison, we will run a small pilot to choose the search budget, number of restarts, and simulated annealing temperature and cooling schedule. Both methods will receive the same maximum number of model fits per run. Previously evaluated subsets will be stored and reused without fitting the same model again, and a separate limit on search steps will prevent repeated visits from extending a run indefinitely. The final settings and stopping rules will be documented in our report.

We will run each method five times with different random seeds to check how consistently it finds useful feature sets. The data split will stay the same across runs. Implementation will use Python, pandas, and scikit-learn, with our own hill climbing and simulated annealing functions. All feature selection and adjustments to the search settings will use training and validation data. Once those choices are complete, we will compare the baseline and selected models on the untouched test set.

# Evaluation Metrics

Brier score will be our main metric for comparing the baseline with the models using selected features. We chose it because we want to evaluate rearrest probabilities, including how confident the model is when it makes an incorrect prediction. Validation Brier score will guide feature selection, and test-set Brier score will measure final performance. It is calculated as:

$$
\text{Brier score} = \frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)^2
$$

$N$ is the number of observations, $p_i$ is the predicted rearrest probability, and $y_i$ is the recorded outcome, either 0 or 1. So, lower Brier scores will indicate that our predicted probabilities are closer to the recorded outcomes.

We will report the area under the receiver operating characteristic curve (ROC-AUC) alongside the Brier score. ROC-AUC measures how well the model ranks people with the outcome above those without it. The ROC curve plots the true-positive rate, $\mathrm{TPR}(u)$, against the false-positive rate, $u$, as the classification threshold changes. The formula below gives the area under that curve. A score of 0.50 indicates chance-level ranking, while 1.00 indicates perfect ranking.

$$
\mathrm{ROC\text{-}AUC} = \int_0^1 \mathrm{TPR}(u)\,du
$$

For the group comparison, we will calculate false-positive rates separately for Black and White individuals using a fixed probability threshold of 0.50. We will report both rates, their absolute difference, and the number of people in each group. Incorrectly predicting rearrest could have consequences if a similar system were used in practice, so we want to examine whether these errors occur more often for one group. For each group, the calculation will be:

$$
\text{FPR} = \frac{FP}{FP+TN}
$$

$FP$ counts people without a recorded rearrest who receive a positive prediction, while $TN$ counts those correctly given a negative prediction. 

For each search method, we will record the number of selected predictors and the search runtime on the same computing environment, then report the average results and standard deviations across the five runs. The false-positive-rate comparison between groups will cover one aspect of fairness.

# Ethics & Privacy

If your project has obvious potential concerns with ethics or data privacy discuss that here.  Almost every ML project put into production can have ethical implications if you use your imagination. Use your imagination. Get creative!

Even if you can't come up with an obvious ethical concern that should be addressed, you should know that a large number of ML projects that go into producation have unintended consequences and ethical problems once in production. How will your team address these issues?

Consider a tool to help you address the potential issues such as https://deon.drivendata.org

# Team Expectations 


* **Communication**: We will use our group chat as our main form of communication and keep each other updated on our progress. And we expected each of us to reply in a reasonable amount of time especially when deadlines are approaching.
* **Meetings**: We will meet regularly to discuss our progress, work, and agress any problems. Everyone is expected to show up for these meetings and come prepared.
* **Availability**: We understand that everyone has different class schedules and other commitments so we will be flexible when scheduling meetings and setting deadlines.
* **Contribution**: Each member is expected to contribute fairly and equally to the project, including research, coding, data analysis, and writing. We will divide responsibilities making sure the workload is balanced. Everyone is expected to contribute ideas, ask questions, and participate in discussions rather than leaving decisions to only one or two members.
* **Handling Conflicts**: If disagreements or problems arise we will communicate respectfully and address them as a group before they affect our progress. If an issue cannot be resolved internally we will reach out to the TA or professor for help.
* **Shared Understanding**: All team members should understand the overall project including our research question, dataset, algorithms, and evaluation metrics, even if they are responsible for different parts.
* **GitHub Collaboration**: We will use our shared GitHub repository to organize project files, track changes, and collaborate on code. Members should commit their changes regularly and communicate before making major modifications to shared files.
* **Quality of Work**: Everyone is responsible for reviewing their contributions and making sure its their own work. We will also review the final code, results, and report together to ensure accuracy, consistency, and just help each other double check the work.

# Project Timeline Proposal


| Meeting Date  | Meeting Time| Completed Before Meeting  | Discuss at Meeting |
|---|---|---|---|
| 10/28  |  5 PM |  Brainstorm potential project topics  | Introduction meeting, discuss communication methods, and brainstorm project ideas  | 
| 11/3  |  3 PM |  Pick topic/dataset and create GitHub repository | Discussed the finalized project topic and ideal dataset, assigned group members to lead each specific part and set a deadline for the work given to each | 
| 11/4  | Before 11:59 PM  | Complete assigned proposal sections  | Review, finalize, and submit proposal   |
| 11/11  | TBD  | Download dataset, review codebook, clean data, and begin exploratory data analysis (EDA) | Review dataset, missing values, feature distributions, and preprocessing decisions   |
| 11/18  | TBD  | Finish preprocessing and implement baseline prediction model | Review baseline results and plan feature-selection algorithms |
| 11/25  | TBD | Begin implementing and testing the selected algorithms | Review progress, discuss coding challenges, and decide on any adjustments needed |
| 12/2  | TBD  | Complete model implementation and begin evaluating results | Compare model performance, discuss findings, and divide up the final report sections  |
| 12/9  | TBD  | Complete data analysis, figures, and first draft of the final report | Review the full project, make final revisions, and ensure everything is ready for submission  |
| 12/11  | Before 11:59 PM  | Finalize code, report, and GitHub repository | Turn in Final Project  |

# Footnotes
<a id="note1"></a>1. Blumstein, A., & Nakamura, K. (2009). “Redemption” in an era of widespread criminal background checks. NIJ Journal, 263, 10–17. https://www.ojp.gov/ncjrs/virtual-library/abstracts/redemption-era-widespread-criminal-background-checks

<a id="note2"></a>2. Pager, D. (2003). The mark of a criminal record. American Journal of Sociology, 108(5), 937–975. https://doi.org/10.1086/374403

<a id="note3"></a>3. Durose, M. R., & Antenucci, L. (2021). Recidivism of Prisoners Released in 34 States in 2012: A 5-Year Follow-Up Period (2012–2017). Bureau of Justice Statistics, NCJ 255947. https://bjs.ojp.gov/library/publications/recidivism-prisoners-released-34-states-2012-5-year-follow-period-2012-2017

